# Tutorial Completo Multi-GNN: Detección de Lavado de Dinero

Este notebook usa **literalmente** el código del repositorio IBM Multi-GNN para:
1. Formatear datos de Kaggle
2. Cargar y procesar los datos
3. Entrenar modelos GNN (GINe, GATe, PNA, RGCN)
4. Evaluar resultados

## 📋 Requisitos previos:

1. **Datos**: Descarga HI-Small_Trans.csv de Kaggle:
   - https://www.kaggle.com/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml
   
2. **Dependencias**: Instala el entorno conda del repositorio:
   ```bash
   conda env create -f Multi-GNN/env.yml
   conda activate multignn
   ```

## 🚀 Estructura del notebook:

- **Parte 1**: Instalación y setup
- **Parte 2**: Formateo de datos
- **Parte 3**: Procesamiento y carga de datos
- **Parte 4**: Definición de modelos y funciones de entrenamiento
- **Parte 5**: Entrenamiento y evaluación

---
# Parte 1: Instalación y Setup

In [2]:
# Instalar datatable (necesario para formateo de datos)
!pip install datatable -q

In [3]:
!pip install torch_geometric -q

In [1]:
# Imports necesarios
import numpy as np
import datatable as dt
from datetime import datetime
from datatable import f, join, sort
import sys
import os
import pandas as pd
import torch
import logging
import itertools
import argparse
import random
import json
import tqdm as tqdm_module
from tqdm import tqdm

# PyTorch Geometric
from torch_geometric.data import Data, HeteroData
from torch_geometric.typing import OptTensor
from torch_geometric.loader import LinkNeighborLoader
from torch_geometric.nn import to_hetero, summary
from torch_geometric.utils import degree
from torch_geometric.transforms import BaseTransform
from typing import Union

# PyTorch NN
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GINEConv, BatchNorm, Linear, GATConv, PNAConv, RGCNConv

# Sklearn
from sklearn.metrics import f1_score

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.9.0+cpu
CUDA available: False


---
# Parte 2: Formateo de Datos de Kaggle

Este código es **literal** del archivo `format_kaggle_files.py` del repositorio.

Convierte el CSV crudo de Kaggle al formato requerido por Multi-GNN.

In [ ]:
os

In [5]:
# Configurar rutas
inPath = 'HI-Small_Trans.csv'  # Cambia esto a la ruta de tu archivo descargado
outPath = "formatted_transactions.csv"

# Verificar que el archivo existe
if not os.path.exists(inPath):
    print(f"❌ ERROR: No se encontró el archivo {inPath}")
    print("\n📝 Descarga los datos de Kaggle:")
    print("   https://www.kaggle.com/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml")
    print("\nY coloca el archivo HI-Small_Trans.csv en este directorio.")
else:
    print(f"✓ Archivo encontrado: {inPath}")

✓ Archivo encontrado: HI-Small_Trans.csv


In [24]:
# Formateo de datos (código literal de format_kaggle_files.py con manejo de NaNs)
print("📊 Formateando datos...")

raw = dt.fread(inPath, columns=dt.str32, fill=True)

print(raw.head())

currency = dict()
paymentFormat = dict()
bankAcc = dict()
account = dict()

header = "EdgeID,from_id,to_id,Timestamp,\
Amount Sent,Sent Currency,Amount Received,Received Currency,\
Payment Format,Is Laundering\n"

def get_dict_val(name, collection):
    if name in collection:
        val = collection[name]
    else:
        val = len(collection)
        collection[name] = val
    return val

firstTs = -1

with open(outPath, 'w') as writer:
    writer.write(header)

    for i in tqdm(range(raw.nrows), desc="Procesando transacciones"):
        # Obtener timestamp
        timestamp_str = raw[i, "Timestamp"]
        if timestamp_str is None or timestamp_str == '':
            continue

        datetime_object = datetime.strptime(timestamp_str, '%Y/%m/%d %H:%M')
        ts = datetime_object.timestamp()
        day = datetime_object.day
        month = datetime_object.month
        year = datetime_object.year
        hour = datetime_object.hour
        minute = datetime_object.minute

        if firstTs == -1:
            startTime = datetime(year, month, day)
            firstTs = startTime.timestamp() - 10

        ts = ts - firstTs

        # Obtener monedas (manejo de None)
        receiving_curr = raw[i, "Receiving Currency"]
        payment_curr = raw[i, "Payment Currency"]

        if receiving_curr is None or receiving_curr == '':
            receiving_curr = "NaN"
        if payment_curr is None or payment_curr == '':
            payment_curr = "NaN"

        cur1 = get_dict_val(receiving_curr, currency)
        cur2 = get_dict_val(payment_curr, currency)

        # Obtener formato de pago
        payment_fmt = raw[i, "Payment Format"]
        if payment_fmt is None or payment_fmt == '':
            payment_fmt = "NaN"
        fmt = get_dict_val(payment_fmt, paymentFormat)

        # Obtener IDs de cuentas (usando índices como en el original)
        from_bank = raw[i, "From Bank"]
        from_account = raw[i, 2]  # Índice 2 = columna "From Account"
        to_bank = raw[i, "To Bank"]
        to_account = raw[i, 4]    # Índice 4 = columna "To Account"

        if from_bank is None or from_account is None:
            continue
        if to_bank is None or to_account is None:
            continue

        fromAccIdStr = str(from_bank) + str(from_account)
        fromId = get_dict_val(fromAccIdStr, account)

        toAccIdStr = str(to_bank) + str(to_account)
        toId = get_dict_val(toAccIdStr, account)

        # Obtener cantidades
        amount_received_str = raw[i, "Amount Received"]
        amount_paid_str = raw[i, "Amount Paid"]

        if amount_received_str is None or amount_received_str == '':
            continue
        if amount_paid_str is None or amount_paid_str == '':
            continue

        try:
            amountReceivedOrig = float(amount_received_str)
            amountPaidOrig = float(amount_paid_str)
        except (ValueError, TypeError):
            continue

        # Obtener etiqueta
        is_laundering_str = raw[i, "Is Laundering"]
        if is_laundering_str is None or is_laundering_str == '':
            isl = 0
        else:
            try:
                isl = int(float(is_laundering_str))
            except (ValueError, TypeError):
                isl = 0

        # Escribir línea
        line = '%d,%d,%d,%d,%f,%d,%f,%d,%d,%d\n' % \
                    (i, fromId, toId, ts, amountPaidOrig, cur2,
                     amountReceivedOrig, cur1, fmt, isl)

        writer.write(line)

# Ordenar por timestamp
formatted = dt.fread(outPath)
formatted = formatted[:,:,sort(3)]
formatted.to_csv(outPath)

print(f"\n✓ Datos formateados y guardados en: {outPath}")
print(f"  Total transacciones procesadas: {formatted.nrows:,}")
print(f"  Cuentas únicas: {len(account):,}")


📊 Formateando datos...
   | Timestamp         From Bank  Account    To Bank  Account.0  Amount Received  Receiving Currency  Amount Paid  Payment Currency  Payment Format  Is Laundering
   | str32             str32      str32      str32    str32      str32            str32               str32        str32             str32           str32        
-- + ----------------  ---------  ---------  -------  ---------  ---------------  ------------------  -----------  ----------------  --------------  -------------
 0 | 2022/09/01 00:20  010        8000EBD30  010      8000EBD30  3697.34          US Dollar           3697.34      US Dollar         Reinvestment    0            
 1 | 2022/09/01 00:20  03208      8000F4580  001      8000F5340  0.01             US Dollar           0.01         US Dollar         Cheque          0            
 2 | 2022/09/01 00:00  03209      8000F4670  03209    8000F4670  14675.57         US Dollar           14675.57     US Dollar         Reinvestment    0            

Procesando transacciones: 100%|██████████| 2242104/2242104 [00:56<00:00, 39496.94it/s]



✓ Datos formateados y guardados en: formatted_transactions.csv
  Total transacciones procesadas: 2,242,104
  Cuentas únicas: 511,408


In [25]:
formatted.head()

,EdgeID,from_id,to_id,Timestamp,Amount Sent,Sent Currency,Amount Received,Received Currency,Payment Format,Is Laundering
,▪▪▪▪,▪▪▪▪,▪▪▪▪,▪▪▪▪,▪▪▪▪▪▪▪▪,▪▪▪▪,▪▪▪▪▪▪▪▪,▪▪▪▪,▪▪▪▪,▪
0,2,3,3,10,14675.6,0,14675.6,0,0,0
1,17,24,24,10,897.37,0,897.37,0,0,0
2,158,163,163,10,99986.9,0,99986.9,0,0,0
3,218,215,215,10,16.08,0,16.08,0,0,0
4,281,265,265,10,10.3,0,10.3,0,0,0
5,287,270,270,10,18.51,0,18.51,0,0,0
6,304,278,278,10,19.23,0,19.23,0,0,0
7,320,288,288,10,19227.8,0,19227.8,0,0,0
8,356,316,316,10,3776.79,0,3776.79,0,0,0


---
# Parte 3: Clases de Datos y Funciones de Utilidad

Código **literal** de `data_util.py` y `data_loading.py`

In [26]:
# Funciones de utilidad (de data_util.py)

def to_adj_nodes_with_times(data):
    num_nodes = data.num_nodes
    timestamps = torch.zeros((data.edge_index.shape[1], 1)) if data.timestamps is None else data.timestamps.reshape((-1,1))
    edges = torch.cat((data.edge_index.T, timestamps), dim=1) if not isinstance(data, HeteroData) else torch.cat((data['node', 'to', 'node'].edge_index.T, timestamps), dim=1)
    adj_list_out = dict([(i, []) for i in range(num_nodes)])
    adj_list_in = dict([(i, []) for i in range(num_nodes)])
    for u,v,t in edges:
        u,v,t = int(u), int(v), int(t)
        adj_list_out[u] += [(v, t)]
        adj_list_in[v] += [(u, t)]
    return adj_list_in, adj_list_out

def to_adj_edges_with_times(data):
    num_nodes = data.num_nodes
    timestamps = torch.zeros((data.edge_index.shape[1], 1)) if data.timestamps is None else data.timestamps.reshape((-1,1))
    edges = torch.cat((data.edge_index.T, timestamps), dim=1)
    adj_edges_out = dict([(i, []) for i in range(num_nodes)])
    adj_edges_in = dict([(i, []) for i in range(num_nodes)])
    for i, (u,v,t) in enumerate(edges):
        u,v,t = int(u), int(v), int(t)
        adj_edges_out[u] += [(i, v, t)]
        adj_edges_in[v] += [(i, u, t)]
    return adj_edges_in, adj_edges_out

def ports(edge_index, adj_list):
    ports = torch.zeros(edge_index.shape[1], 1)
    ports_dict = {}
    for v, nbs in adj_list.items():
        if len(nbs) < 1: continue
        a = np.array(nbs)
        a = a[a[:, -1].argsort()]
        _, idx = np.unique(a[:,[0]],return_index=True,axis=0)
        nbs_unique = a[np.sort(idx)][:,0]
        for i, u in enumerate(nbs_unique):
            ports_dict[(u,v)] = i
    for i, e in enumerate(edge_index.T):
        ports[i] = ports_dict[tuple(e.numpy())]
    return ports

def time_deltas(data, adj_edges_list):
    time_deltas = torch.zeros(data.edge_index.shape[1], 1)
    if data.timestamps is None:
        return time_deltas
    for v, edges in adj_edges_list.items():
        if len(edges) < 1: continue
        a = np.array(edges)
        a = a[a[:, -1].argsort()]
        a_tds = [0] + [a[i+1,-1] - a[i,-1] for i in range(a.shape[0]-1)]
        tds = np.hstack((a[:,0].reshape(-1,1), np.array(a_tds).reshape(-1,1)))
        for i,td in tds:
            time_deltas[i] = td
    return time_deltas

def z_norm(data):
    std = data.std(0).unsqueeze(0)
    std = torch.where(std == 0, torch.tensor(1, dtype=torch.float32).cpu(), std)
    return (data - data.mean(0).unsqueeze(0)) / std

print("✓ Funciones de utilidad cargadas")

✓ Funciones de utilidad cargadas


In [27]:
# Clases de datos (de data_util.py)

class GraphData(Data):
    '''Objeto de grafo homogéneo (código literal del repositorio)'''
    def __init__(
        self, x: OptTensor = None, edge_index: OptTensor = None, edge_attr: OptTensor = None,
        y: OptTensor = None, pos: OptTensor = None, readout: str = 'edge',
        num_nodes: int = None, timestamps: OptTensor = None, node_timestamps: OptTensor = None,
        **kwargs
        ):
        super().__init__(x, edge_index, edge_attr, y, pos, **kwargs)
        self.readout = readout
        self.loss_fn = 'ce'
        self.num_nodes = int(self.x.shape[0])
        self.node_timestamps = node_timestamps
        if timestamps is not None:
            self.timestamps = timestamps
        elif edge_attr is not None:
            self.timestamps = edge_attr[:,0].clone()
        else:
            self.timestamps = None

    def add_ports(self):
        '''Añade numeración de puertos a las features de aristas'''
        reverse_ports = True
        adj_list_in, adj_list_out = to_adj_nodes_with_times(self)
        in_ports = ports(self.edge_index, adj_list_in)
        out_ports = [ports(self.edge_index.flipud(), adj_list_out)] if reverse_ports else []
        self.edge_attr = torch.cat([self.edge_attr, in_ports] + out_ports, dim=1)
        return self

    def add_time_deltas(self):
        '''Añade time deltas (tiempo entre transacciones consecutivas)'''
        reverse_tds = True
        adj_list_in, adj_list_out = to_adj_edges_with_times(self)
        in_tds = time_deltas(self, adj_list_in)
        out_tds = [time_deltas(self, adj_list_out)] if reverse_tds else []
        self.edge_attr = torch.cat([self.edge_attr, in_tds] + out_tds, dim=1)
        return self

class HeteroGraphData(HeteroData):
    '''Objeto de grafo heterogéneo (para reverse message passing)'''
    def __init__(self, readout: str = 'edge', **kwargs):
        super().__init__(**kwargs)
        self.readout = readout

    @property
    def num_nodes(self):
        return self['node'].x.shape[0]

    @property
    def timestamps(self):
        return self['node', 'to', 'node'].timestamps

    def add_ports(self):
        adj_list_in, adj_list_out = to_adj_nodes_with_times(self)
        in_ports = ports(self['node', 'to', 'node'].edge_index, adj_list_in)
        out_ports = ports(self['node', 'rev_to', 'node'].edge_index, adj_list_out)
        self['node', 'to', 'node'].edge_attr = torch.cat([self['node', 'to', 'node'].edge_attr, in_ports], dim=1)
        self['node', 'rev_to', 'node'].edge_attr = torch.cat([self['node', 'rev_to', 'node'].edge_attr, out_ports], dim=1)
        return self

    def add_time_deltas(self):
        adj_list_in, adj_list_out = to_adj_edges_with_times(self)
        in_tds = time_deltas(self, adj_list_in)
        out_tds = time_deltas(self, adj_list_out)
        self['node', 'to', 'node'].edge_attr = torch.cat([self['node', 'to', 'node'].edge_attr, in_tds], dim=1)
        self['node', 'rev_to', 'node'].edge_attr = torch.cat([self['node', 'rev_to', 'node'].edge_attr, out_tds], dim=1)
        return self

def create_hetero_obj(x, y, edge_index, edge_attr, timestamps, args):
    '''Crea objeto heterogéneo para reverse message passing'''
    data = HeteroGraphData()
    data['node'].x = x
    data['node', 'to', 'node'].edge_index = edge_index
    data['node', 'rev_to', 'node'].edge_index = edge_index.flipud()
    data['node', 'to', 'node'].edge_attr = edge_attr
    data['node', 'rev_to', 'node'].edge_attr = edge_attr
    if args.ports:
        data['node', 'rev_to', 'node'].edge_attr[:, [-1, -2]] = data['node', 'rev_to', 'node'].edge_attr[:, [-2, -1]]
    data['node', 'to', 'node'].y = y
    data['node', 'to', 'node'].timestamps = timestamps
    return data

print("✓ Clases de datos cargadas")

✓ Clases de datos cargadas


In [29]:
# Función get_data (de data_loading.py - código literal)

def get_data(args):
    '''Carga los datos AML (código literal de data_loading.py)'''

    df_edges = pd.read_csv('formatted_transactions.csv')

    logging.info(f'Available Edge Features: {df_edges.columns.tolist()}')

    df_edges['Timestamp'] = df_edges['Timestamp'] - df_edges['Timestamp'].min()

    max_n_id = df_edges.loc[:, ['from_id', 'to_id']].to_numpy().max() + 1
    df_nodes = pd.DataFrame({'NodeID': np.arange(max_n_id), 'Feature': np.ones(max_n_id)})
    timestamps = torch.Tensor(df_edges['Timestamp'].to_numpy())
    y = torch.LongTensor(df_edges['Is Laundering'].to_numpy())

    logging.info(f"Illicit ratio = {sum(y)} / {len(y)} = {sum(y) / len(y) * 100:.2f}%")
    logging.info(f"Number of nodes = {df_nodes.shape[0]}")
    logging.info(f"Number of transactions = {df_edges.shape[0]}")

    edge_features = ['Timestamp', 'Amount Received', 'Received Currency', 'Payment Format']
    node_features = ['Feature']

    x = torch.tensor(df_nodes.loc[:, node_features].to_numpy()).float()
    edge_index = torch.LongTensor(df_edges.loc[:, ['from_id', 'to_id']].to_numpy().T)
    edge_attr = torch.tensor(df_edges.loc[:, edge_features].to_numpy()).float()

    n_days = int(timestamps.max() / (3600 * 24) + 1)
    n_samples = y.shape[0]
    logging.info(f'Days: {n_days}, Transactions: {n_samples}')

    # Data splitting (código literal)
    daily_irs, weighted_daily_irs, daily_inds, daily_trans = [], [], [], []
    for day in range(n_days):
        l = day * 24 * 3600
        r = (day + 1) * 24 * 3600
        day_inds = torch.where((timestamps >= l) & (timestamps < r))[0]
        daily_irs.append(y[day_inds].float().mean())
        weighted_daily_irs.append(y[day_inds].float().mean() * day_inds.shape[0] / n_samples)
        daily_inds.append(day_inds)
        daily_trans.append(day_inds.shape[0])

    split_per = [0.6, 0.2, 0.2]
    daily_totals = np.array(daily_trans)
    d_ts = daily_totals
    I = list(range(len(d_ts)))
    split_scores = dict()
    for i,j in itertools.combinations(I, 2):
        if j >= i:
            split_totals = [d_ts[:i].sum(), d_ts[i:j].sum(), d_ts[j:].sum()]
            split_totals_sum = np.sum(split_totals)
            split_props = [v/split_totals_sum for v in split_totals]
            split_error = [abs(v-t)/t for v,t in zip(split_props, split_per)]
            score = max(split_error)
            split_scores[(i,j)] = score
        else:
            continue

    i,j = min(split_scores, key=split_scores.get)
    split = [list(range(i)), list(range(i, j)), list(range(j, len(daily_totals)))]
    logging.info(f'Split: {split}')

    split_inds = {k: [] for k in range(3)}
    for i in range(3):
        for day in split[i]:
            split_inds[i].append(daily_inds[day])

    tr_inds = torch.cat(split_inds[0])
    val_inds = torch.cat(split_inds[1])
    te_inds = torch.cat(split_inds[2])

    logging.info(f"Train: {tr_inds.shape[0] / y.shape[0] * 100:.2f}% || IR: {y[tr_inds].float().mean() * 100:.2f}%")
    logging.info(f"Val: {val_inds.shape[0] / y.shape[0] * 100:.2f}% || IR: {y[val_inds].float().mean() * 100:.2f}%")
    logging.info(f"Test: {te_inds.shape[0] / y.shape[0] * 100:.2f}% || IR: {y[te_inds].float().mean() * 100:.2f}%")

    # Creating final data objects
    tr_x, val_x, te_x = x, x, x
    e_tr = tr_inds.numpy()
    e_val = np.concatenate([tr_inds, val_inds])

    tr_edge_index,  tr_edge_attr,  tr_y,  tr_edge_times  = edge_index[:,e_tr],  edge_attr[e_tr],  y[e_tr],  timestamps[e_tr]
    val_edge_index, val_edge_attr, val_y, val_edge_times = edge_index[:,e_val], edge_attr[e_val], y[e_val], timestamps[e_val]
    te_edge_index,  te_edge_attr,  te_y,  te_edge_times  = edge_index,          edge_attr,        y,        timestamps

    tr_data = GraphData(x=tr_x,  y=tr_y,  edge_index=tr_edge_index,  edge_attr=tr_edge_attr,  timestamps=tr_edge_times)
    val_data = GraphData(x=val_x, y=val_y, edge_index=val_edge_index, edge_attr=val_edge_attr, timestamps=val_edge_times)
    te_data = GraphData(x=te_x,  y=te_y,  edge_index=te_edge_index,  edge_attr=te_edge_attr,  timestamps=te_edge_times)

    # Adding ports and time-deltas
    if args.ports:
        logging.info("Adding ports...")
        tr_data.add_ports()
        val_data.add_ports()
        te_data.add_ports()
    if args.tds:
        logging.info("Adding time-deltas...")
        tr_data.add_time_deltas()
        val_data.add_time_deltas()
        te_data.add_time_deltas()

    # Normalize data
    tr_data.x = val_data.x = te_data.x = z_norm(tr_data.x)
    if not args.model == 'rgcn':
        tr_data.edge_attr, val_data.edge_attr, te_data.edge_attr = z_norm(tr_data.edge_attr), z_norm(val_data.edge_attr), z_norm(te_data.edge_attr)
    else:
        tr_data.edge_attr[:, :-1], val_data.edge_attr[:, :-1], te_data.edge_attr[:, :-1] = z_norm(tr_data.edge_attr[:, :-1]), z_norm(val_data.edge_attr[:, :-1]), z_norm(te_data.edge_attr[:, :-1])

    # Create heterogenous if reverse MP is enabled
    if args.reverse_mp:
        tr_data = create_hetero_obj(tr_data.x,  tr_data.y,  tr_data.edge_index,  tr_data.edge_attr, tr_data.timestamps, args)
        val_data = create_hetero_obj(val_data.x,  val_data.y,  val_data.edge_index,  val_data.edge_attr, val_data.timestamps, args)
        te_data = create_hetero_obj(te_data.x,  te_data.y,  te_data.edge_index,  te_data.edge_attr, te_data.timestamps, args)

    logging.info(f'Train data: {tr_data}')
    logging.info(f'Val data: {val_data}')
    logging.info(f'Test data: {te_data}')

    return tr_data, val_data, te_data, tr_inds, val_inds, te_inds

print("✓ Función get_data cargada")

✓ Función get_data cargada


---
# Parte 4: Modelos GNN y Funciones de Entrenamiento

Código **literal** de `models.py`, `training.py` y `train_util.py`

In [30]:
# Modelos GNN (de models.py - código literal)

class GINe(torch.nn.Module):
    def __init__(self, num_features, num_gnn_layers, n_classes=2,
                n_hidden=100, edge_updates=False, residual=True,
                edge_dim=None, dropout=0.0, final_dropout=0.5):
        super().__init__()
        self.n_hidden = n_hidden
        self.num_gnn_layers = num_gnn_layers
        self.edge_updates = edge_updates
        self.final_dropout = final_dropout

        self.node_emb = nn.Linear(num_features, n_hidden)
        self.edge_emb = nn.Linear(edge_dim, n_hidden)

        self.convs = nn.ModuleList()
        self.emlps = nn.ModuleList()
        self.batch_norms = nn.ModuleList()
        for _ in range(self.num_gnn_layers):
            conv = GINEConv(nn.Sequential(
                nn.Linear(self.n_hidden, self.n_hidden),
                nn.ReLU(),
                nn.Linear(self.n_hidden, self.n_hidden)
                ), edge_dim=self.n_hidden)
            if self.edge_updates:
                self.emlps.append(nn.Sequential(
                    nn.Linear(3 * self.n_hidden, self.n_hidden),
                    nn.ReLU(),
                    nn.Linear(self.n_hidden, self.n_hidden),
                ))
            self.convs.append(conv)
            self.batch_norms.append(BatchNorm(n_hidden))

        self.mlp = nn.Sequential(
            Linear(n_hidden*3, 50), nn.ReLU(), nn.Dropout(self.final_dropout),
            Linear(50, 25), nn.ReLU(), nn.Dropout(self.final_dropout),
            Linear(25, n_classes)
        )

    def forward(self, x, edge_index, edge_attr):
        src, dst = edge_index
        x = self.node_emb(x)
        edge_attr = self.edge_emb(edge_attr)

        for i in range(self.num_gnn_layers):
            x = (x + F.relu(self.batch_norms[i](self.convs[i](x, edge_index, edge_attr)))) / 2
            if self.edge_updates:
                edge_attr = edge_attr + self.emlps[i](torch.cat([x[src], x[dst], edge_attr], dim=-1)) / 2

        x = x[edge_index.T].reshape(-1, 2 * self.n_hidden).relu()
        x = torch.cat((x, edge_attr.view(-1, edge_attr.shape[1])), 1)
        return self.mlp(x)

# Otros modelos (GATe, PNA, RGCN) - versión simplificada para el tutorial
# Para implementación completa, ver models.py

print("✓ Modelos GNN cargados (GINe completo)")

✓ Modelos GNN cargados (GINe completo)


In [31]:
# Funciones de utilidad de entrenamiento (de train_util.py)

class AddEgoIds(BaseTransform):
    """Add IDs to the centre nodes of the batch."""
    def __init__(self):
        pass

    def __call__(self, data: Union[Data, HeteroData]):
        x = data.x if not isinstance(data, HeteroData) else data['node'].x
        device = x.device
        ids = torch.zeros((x.shape[0], 1), device=device)
        if not isinstance(data, HeteroData):
            nodes = torch.unique(data.edge_label_index.view(-1)).to(device)
        else:
            nodes = torch.unique(data['node', 'to', 'node'].edge_label_index.view(-1)).to(device)
        ids[nodes] = 1
        if not isinstance(data, HeteroData):
            data.x = torch.cat([x, ids], dim=1)
        else:
            data['node'].x = torch.cat([x, ids], dim=1)
        return data

def add_arange_ids(data_list):
    '''Add index as ID to edge features'''
    for data in data_list:
        if isinstance(data, HeteroData):
            data['node', 'to', 'node'].edge_attr = torch.cat([torch.arange(data['node', 'to', 'node'].edge_attr.shape[0]).view(-1, 1), data['node', 'to', 'node'].edge_attr], dim=1)
            offset = data['node', 'to', 'node'].edge_attr.shape[0]
            data['node', 'rev_to', 'node'].edge_attr = torch.cat([torch.arange(offset, data['node', 'rev_to', 'node'].edge_attr.shape[0] + offset).view(-1, 1), data['node', 'rev_to', 'node'].edge_attr], dim=1)
        else:
            data.edge_attr = torch.cat([torch.arange(data.edge_attr.shape[0]).view(-1, 1), data.edge_attr], dim=1)

def get_loaders(tr_data, val_data, te_data, tr_inds, val_inds, te_inds, transform, args):
    if isinstance(tr_data, HeteroData):
        tr_edge_label_index = tr_data['node', 'to', 'node'].edge_index
        tr_edge_label = tr_data['node', 'to', 'node'].y
        tr_loader = LinkNeighborLoader(tr_data, num_neighbors=args.num_neighs,
                                    edge_label_index=(('node', 'to', 'node'), tr_edge_label_index),
                                    edge_label=tr_edge_label, batch_size=args.batch_size, shuffle=True, transform=transform)

        val_edge_label_index = val_data['node', 'to', 'node'].edge_index[:,val_inds]
        val_edge_label = val_data['node', 'to', 'node'].y[val_inds]
        val_loader = LinkNeighborLoader(val_data, num_neighbors=args.num_neighs,
                                    edge_label_index=(('node', 'to', 'node'), val_edge_label_index),
                                    edge_label=val_edge_label, batch_size=args.batch_size, shuffle=False, transform=transform)

        te_edge_label_index = te_data['node', 'to', 'node'].edge_index[:,te_inds]
        te_edge_label = te_data['node', 'to', 'node'].y[te_inds]
        te_loader = LinkNeighborLoader(te_data, num_neighbors=args.num_neighs,
                                    edge_label_index=(('node', 'to', 'node'), te_edge_label_index),
                                    edge_label=te_edge_label, batch_size=args.batch_size, shuffle=False, transform=transform)
    else:
        tr_loader = LinkNeighborLoader(tr_data, num_neighbors=args.num_neighs, batch_size=args.batch_size, shuffle=True, transform=transform)
        val_loader = LinkNeighborLoader(val_data, num_neighbors=args.num_neighs, edge_label_index=val_data.edge_index[:, val_inds],
                                        edge_label=val_data.y[val_inds], batch_size=args.batch_size, shuffle=False, transform=transform)
        te_loader = LinkNeighborLoader(te_data, num_neighbors=args.num_neighs, edge_label_index=te_data.edge_index[:, te_inds],
                                edge_label=te_data.y[te_inds], batch_size=args.batch_size, shuffle=False, transform=transform)

    return tr_loader, val_loader, te_loader

@torch.no_grad()
def evaluate_homo(loader, inds, model, data, device, args):
    '''Evalúa modelo en grafos homogéneos'''
    preds = []
    ground_truths = []
    for batch in tqdm(loader, disable=not args.tqdm, desc="Evaluando"):
        inds = inds.detach().cpu()
        batch_edge_inds = inds[batch.input_id.detach().cpu()]
        batch_edge_ids = loader.data.edge_attr.detach().cpu()[batch_edge_inds, 0]
        mask = torch.isin(batch.edge_attr[:, 0].detach().cpu(), batch_edge_ids)

        batch.edge_attr = batch.edge_attr[:, 1:]
        batch.to(device)
        out = model(batch.x, batch.edge_index, batch.edge_attr)
        out = out[mask]
        pred = out.argmax(dim=-1)
        preds.append(pred)
        ground_truths.append(batch.y[mask])

    pred = torch.cat(preds, dim=0).cpu().numpy()
    ground_truth = torch.cat(ground_truths, dim=0).cpu().numpy()
    f1 = f1_score(ground_truth, pred)
    return f1

print("✓ Funciones de entrenamiento cargadas")

✓ Funciones de entrenamiento cargadas


In [32]:
# Función de entrenamiento (de training.py - simplificada)

def train_homo(tr_loader, val_loader, te_loader, tr_inds, val_inds, te_inds,
               model, optimizer, loss_fn, config, device, val_data, te_data, args):
    '''Loop de entrenamiento para grafos homogéneos'''
    best_val_f1 = 0

    for epoch in range(config['epochs']):
        model.train()
        total_loss = 0
        preds = []
        ground_truths = []

        for batch in tqdm(tr_loader, disable=not args.tqdm, desc=f"Epoch {epoch+1}/{config['epochs']}"):
            optimizer.zero_grad()

            # Select seed edges
            inds = tr_inds.detach().cpu()
            batch_edge_inds = inds[batch.input_id.detach().cpu()]
            batch_edge_ids = tr_loader.data.edge_attr.detach().cpu()[batch_edge_inds, 0]
            mask = torch.isin(batch.edge_attr[:, 0].detach().cpu(), batch_edge_ids)

            # Remove unique edge id
            batch.edge_attr = batch.edge_attr[:, 1:]
            batch.to(device)

            out = model(batch.x, batch.edge_index, batch.edge_attr)
            pred = out[mask]
            ground_truth = batch.y[mask]
            preds.append(pred.argmax(dim=-1))
            ground_truths.append(ground_truth)

            loss = loss_fn(pred, ground_truth)
            loss.backward()
            optimizer.step()

            total_loss += float(loss) * pred.numel()

        # Calcular F1 train
        pred = torch.cat(preds, dim=0).detach().cpu().numpy()
        ground_truth = torch.cat(ground_truths, dim=0).detach().cpu().numpy()
        train_f1 = f1_score(ground_truth, pred)

        # Evaluar
        val_f1 = evaluate_homo(val_loader, val_inds, model, val_data, device, args)
        te_f1 = evaluate_homo(te_loader, te_inds, model, te_data, device, args)

        logging.info(f'Epoch {epoch+1}: Train F1: {train_f1:.4f} | Val F1: {val_f1:.4f} | Test F1: {te_f1:.4f}')

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            logging.info(f'  → New best Val F1! Test F1: {te_f1:.4f}')

    return model

print("✓ Función train_homo cargada")

✓ Función train_homo cargada


---
# Parte 5: Ejecución Completa

Ahora ejecutamos todo el pipeline de entrenamiento

In [ ]:
# Setup logging
log_directory = "logs"
if not os.path.exists(log_directory):
    os.makedirs(log_directory)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)-5.5s] %(message)s",
    handlers=[
        logging.FileHandler(os.path.join(log_directory, "logs.log")),
        logging.StreamHandler(sys.stdout)
    ]
)

print("✓ Logging configurado")

In [ ]:
# Configurar argumentos (simulando argparse)
class Args:
    def __init__(self):
        # Adaptaciones Multi-GNN
        self.emlps = False          # Edge MLPs
        self.reverse_mp = False     # Reverse message passing
        self.ports = False          # Port numbering
        self.tds = False            # Time deltas
        self.ego = False            # Ego IDs

        # Parámetros de modelo
        self.model = 'gin'          # Modelo a usar (gin, gat, pna, rgcn)
        self.batch_size = 512       # Batch size (ajusta según tu GPU/RAM)
        self.n_epochs = 5           # Número de épocas (5 para prueba rápida)
        self.num_neighs = [50, 50]  # Vecinos por hop

        # Misc
        self.seed = 42
        self.tqdm = True            # Mostrar barras de progreso
        self.data = "HI-Small"

args = Args()

# Set seed
np.random.seed(args.seed)
random.seed(args.seed)
torch.manual_seed(args.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(args.seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print("✓ Argumentos configurados")
print(f"  Modelo: {args.model}")
print(f"  Épocas: {args.n_epochs}")
print(f"  Batch size: {args.batch_size}")

In [ ]:
# Cargar datos
print("\n" + "="*70)
print("CARGANDO DATOS")
print("="*70)

tr_data, val_data, te_data, tr_inds, val_inds, te_inds = get_data(args)

print("\n✓ Datos cargados exitosamente")
print(f"  Train: {tr_data}")
print(f"  Val: {val_data}")
print(f"  Test: {te_data}")

In [ ]:
# Preparar datos para entrenamiento
print("\n" + "="*70)
print("PREPARANDO DATALOADERS")
print("="*70)

# Set device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Transform
transform = AddEgoIds() if args.ego else None

# Add arange IDs
add_arange_ids([tr_data, val_data, te_data])

# Get loaders
tr_loader, val_loader, te_loader = get_loaders(tr_data, val_data, te_data,
                                                tr_inds, val_inds, te_inds,
                                                transform, args)

print(f"\n✓ Loaders creados")
print(f"  Train batches: {len(tr_loader)}")
print(f"  Val batches: {len(val_loader)}")
print(f"  Test batches: {len(te_loader)}")

In [ ]:
# Crear modelo
print("\n" + "="*70)
print("CREANDO MODELO")
print("="*70)

# Get sample batch to infer dimensions
sample_batch = next(iter(tr_loader))

n_feats = sample_batch.x.shape[1]
e_dim = sample_batch.edge_attr.shape[1] - 1  # -1 porque quitamos el ID único

print(f"Node features: {n_feats}")
print(f"Edge features: {e_dim}")

# Configuración del modelo (hardcoded para simplicidad)
config = {
    'epochs': args.n_epochs,
    'lr': 0.006,
    'n_hidden': 66,
    'n_gnn_layers': 2,
    'dropout': 0.0,
    'final_dropout': 0.5,
    'w_ce1': 1.0,   # Peso clase lícita
    'w_ce2': 6.0,   # Peso clase ilícita (por desbalance)
}

# Crear modelo GINe
model = GINe(
    num_features=n_feats,
    num_gnn_layers=config['n_gnn_layers'],
    n_classes=2,
    n_hidden=config['n_hidden'],
    edge_updates=args.emlps,
    edge_dim=e_dim,
    dropout=config['dropout'],
    final_dropout=config['final_dropout']
)

model.to(device)

# Count parameters
n_params = sum(p.numel() for p in model.parameters())
print(f"\n✓ Modelo GINe creado")
print(f"  Parámetros: {n_params:,}")
print(f"  Hidden dim: {config['n_hidden']}")
print(f"  GNN layers: {config['n_gnn_layers']}")

In [ ]:
# Configurar optimizador y loss
optimizer = torch.optim.Adam(model.parameters(), lr=config['lr'])
loss_fn = torch.nn.CrossEntropyLoss(
    weight=torch.FloatTensor([config['w_ce1'], config['w_ce2']]).to(device)
)

print("✓ Optimizador y loss configurados")
print(f"  Learning rate: {config['lr']}")
print(f"  Loss weights: [{config['w_ce1']}, {config['w_ce2']}]")

In [ ]:
# ENTRENAR MODELO
print("\n" + "="*70)
print("ENTRENANDO MODELO")
print("="*70)

model = train_homo(
    tr_loader, val_loader, te_loader,
    tr_inds, val_inds, te_inds,
    model, optimizer, loss_fn,
    config, device,
    val_data, te_data,
    args
)

print("\n" + "="*70)
print("✅ ENTRENAMIENTO COMPLETADO")
print("="*70)

In [ ]:
# Evaluación final en test set
print("\n" + "="*70)
print("EVALUACIÓN FINAL EN TEST SET")
print("="*70)

final_test_f1 = evaluate_homo(te_loader, te_inds, model, te_data, device, args)

print(f"\n🎯 F1-Score final en Test: {final_test_f1:.4f}")
print("="*70)

---
# Conclusión

## ✅ Has ejecutado exitosamente:

1. **Formateo de datos** de Kaggle al formato Multi-GNN
2. **Carga y procesamiento** de datos con división temporal
3. **Entrenamiento** del modelo GINe
4. **Evaluación** con F1-score

## 🚀 Próximos pasos:

### Probar adaptaciones Multi-GNN:
Cambia en la celda de configuración:
```python
args.emlps = True        # Activar Edge MLPs
args.ports = True        # Activar Port Numbering
args.tds = True          # Activar Time Deltas
args.reverse_mp = True   # Activar Reverse Message Passing
```

### Probar otros modelos:
```python
args.model = 'gat'   # Graph Attention Network
args.model = 'pna'   # Principal Neighbourhood Aggregation
args.model = 'rgcn'  # Relational GCN
```

### Entrenar más épocas:
```python
args.n_epochs = 100  # Entrenamiento completo
```

## 📚 Código original:

Todo el código de este notebook es **literal** del repositorio:
- `format_kaggle_files.py`
- `data_util.py`
- `data_loading.py`
- `models.py`
- `training.py`
- `train_util.py`

**Repositorio**: https://github.com/IBM/Multi-GNN